### **Challenge 4** — Human-in-the-Loop Approval Agent

### What to build:
A LangGraph agent that proposes a refund action, pauses for human approval using interrupt(), then either executes or rejects based on human input.

### What it should do:

- User describes a refund scenario
- Agent proposes a specific action ("Refund $X for customer Y")
- Graph pauses — asks human "Approve or reject?"
- Human types "approve" → agent executes and confirms
- Human types "reject" → agent cancels and explains

### Constraints:

- Use interrupt() and Command(resume=...) — not plain input()
- Must use MemorySaver as checkpointer
- Same thread_id must be used for both the first invoke and the resume invoke
- Use TypedDict for state with at least: query, proposed_action, approval, result

In [140]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

True

In [141]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import interrupt, Command
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage
import uuid # for thread_id

In [142]:
model = init_chat_model(
    model="qwen/qwen3.8-27b",       # The specific Groq model ID
    model_provider="groq",        # Specifies the provider
    temperature=0                 # Optional parameters
)

In [143]:
# Define State with required fields: query, proposed_action, approval, result
from typing import Literal
from typing_extensions import TypedDict

class State(TypedDict):
    query: str
    proposed_action: str
    approval: Literal["approve", "reject", "manual_review"]
    result: str
    customer: str
    amount: float

In [144]:
# Node 1: Analyze user query and propose a refund action (approve/reject/manual_review)
def propossedAction(state: State) -> State:
    """Analyze the refund request and propose a specific action."""
    user_request = state["query"]

    prompt = f"""Analyze this refund request: "{user_request}"

Extract the key details and propose a specific action in this format:
"Refund $X for customer Y (reason: Z)"

Decide the appropriate action:
- approve
- reject
- manual_review

Return ONLY this exact format (no extra text):

ACTION: <approve|reject|manual_review>
CUSTOMER: <customer name or ID>
AMOUNT: <numeric amount only, e.g. 150>
REASON: <one-line reason>"""

    response = model.invoke([HumanMessage(content=prompt)])
    proposed = response.content

    return{
        **state,
        "proposed_action": proposed,

    }


In [145]:
# Node 2: Human approval via interrupt
def human_review(state: State) -> Command:
    """Pause the graph and ask human for approval via interrupt()."""
    # this will pause the the execution and wait for  human input via Command(resume=...)
    is_approved = interrupt({
    "question": "Do you want to proceed with this action?",
    "proposed_action": state["proposed_action"]
})

    return Command(
        goto="execute_action",
        update={"approval": is_approved}
    )

In [146]:
def execute_action(state: State) -> State:
    """Execute the refund action."""
    if state['approval'] == "approve":
        result = f" Refund Executed: {state['proposed_action']}"
    else:
        result = f"❌ REFUND CANCELLED: {state['proposed_action']} — Reason: Human rejected the action"

    return {
        **state,
        "result": result
    }
    

In [147]:
builder = StateGraph(State)

# node
builder.add_node("propossed_action",propossedAction)
builder.add_node("human_review", human_review)
builder.add_node("execute_action", execute_action)

# edges
builder.add_edge(START,"propossed_action")
builder.add_edge("propossed_action", "human_review")
builder.add_edge("human_review","execute_action")
builder.add_edge("execute_action", END)

# checkpointer for storing message in RAM for short duration
checkpoint = MemorySaver()
store = InMemoryStore()

graph = builder.compile(checkpointer=checkpoint, store=store)

print("Graph compiled successfully!")
print("Nodes:", list(graph.nodes.keys()))

Graph compiled successfully!
Nodes: ['__start__', 'propossed_action', 'human_review', 'execute_action']


In [148]:
def run():
    """Helper to run a test case with human-in-the-loop approval."""
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    user_request = input("User Request: ")

    initial_state = {
        "query": user_request,
        "proposed_action": "",
        "customer": "",
        "amount": 0.0,
        "approval": "",
        "result": ""
    }

    # First invoke - proposes action and pauses at interrupt
    result = graph.invoke(initial_state, config=config)
    
    print("\n" + "=" * 60)
    print("🤖 AGENT PROPOSAL:")
    print(f"   Proposed Action: {result['proposed_action']}")
    print("=" * 60)
    
    # Get human decision
    human_decision = input("\nHuman Decision (approve/reject): ").strip().lower()
    while human_decision not in ("approve", "reject"):
        human_decision = input("Please enter 'approve' or 'reject': ").strip().lower()
    
    # Resume with human decision
    resume_result = graph.invoke(Command(resume=human_decision), config=config)
    return resume_result

In [149]:
run()


🤖 AGENT PROPOSAL:
   Proposed Action: ACTION: approve
CUSTOMER: Sarah Lee
AMOUNT: 75
REASON: wrong item was shipped


{'query': 'Customer Sarah Lee wants a refund of $75 for order #99001 because the wrong item was shipped',
 'proposed_action': 'ACTION: approve\nCUSTOMER: Sarah Lee\nAMOUNT: 75\nREASON: wrong item was shipped',
 'approval': 'approve',
 'result': ' Refund Executed: ACTION: approve\nCUSTOMER: Sarah Lee\nAMOUNT: 75\nREASON: wrong item was shipped',
 'customer': '',
 'amount': 0.0}